In [1]:
!git clone https://github.com/sht037-lgtm/Q-Vtree.git

Cloning into 'Q-Vtree'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 93 (delta 31), reused 77 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 1.19 MiB | 5.16 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [2]:
# log in HuggingFace
!pip install -U huggingface_hub
!hf auth login --token "YOUR HF TOKEN"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 22.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `ZoomEye` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `ZoomEye`


In [ ]:
# download Qwen2.5-VL-3B-Instruct Model
%cd Q-Vtree/models
!python download.py
!ls

/kaggle/working/Q-Vtree/models
[INFO] Downloading Qwen/Qwen2.5-VL-3B-Instruct ...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:186: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 14 files: 100%|████████████████████████| 14/14 [00:30<00:00,  2.16s/it]
Download complete: : 7.52GB [00:30, 300MB/s]              [INFO] Download complete: /kaggle/working/Q-Vtree/models/Qwen2.5-VL-3B-Instruct
Model path: /kaggle/working/Q-Vtree/models/Qwen2.5-VL-3B-Instruct
Download complete: : 7.52GB [00:30, 248MB/s]
download.py  Qwen2.5-VL-3B-Instruct


In [ ]:
# load Qwen model
from transformers import AutoProcessor
import torch
from qwen_tree import Qwen2_5_VLForConditionalGenerationWithTree

model_path = "models/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(model_path)

model = Qwen2_5_VLForConditionalGenerationWithTree.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "img/qwen_demo.jpeg"},
            {"type": "text", "text": "What is in the image?"}
        ],
    }
]

text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

from qwen_vl_utils import process_vision_info
image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)

device = next(model.parameters()).device
inputs = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in inputs.items()}

In [ ]:
with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=64)

decoded = processor.batch_decode(
    outputs[:, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Output:", decoded)

In [ ]:
# check if selects successfully
print("Selected token indices per image:")
print(model.model._debug_selected_idx)

sel = model.model._debug_selected_idx[0]
print("Selected:", sel.numel())

total_tokens = model.model.get_image_features(
    inputs["pixel_values"],
    inputs["image_grid_thw"],
    return_dict=True
).pooler_output[0].shape[0]

print("Total:", total_tokens)